In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib, os
os.makedirs('./models', exist_ok=True)

# 데이터 로드
train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# layout merge
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

X = train[feature_cols]
y = train[TARGET]

# best iteration 확인 (lr=0.05)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# lr=0.1로 더 높이고 num_leaves도 줄이기
model_step2 = lgb.LGBMRegressor(
    n_estimators     = 100000,
    learning_rate    = 0.1,
    num_leaves       = 128,   # 230 → 128
    max_depth        = 12,
    min_child_samples= 37,
    subsample        = 0.6971592844853077,
    colsample_bytree = 0.6283895747824833,
    reg_alpha        = 0.3936177260538011,
    reg_lambda       = 1.0229688573580245,
    objective        = 'mae',
    metric           = 'mae',
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

model_step2.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(300), lgb.log_evaluation(2000)]
)

best_iter = model_step2.best_iteration_
print(f'Best iteration: {best_iter}')
print(f'Validation MAE: {np.mean(np.abs(y_val - model_step2.predict(X_val))):.4f}')

Training until validation scores don't improve for 300 rounds
[2000]	valid_0's l1: 7.01634
[4000]	valid_0's l1: 6.96465
[6000]	valid_0's l1: 6.92818
[8000]	valid_0's l1: 6.90835
[10000]	valid_0's l1: 6.89423
[12000]	valid_0's l1: 6.88336
[14000]	valid_0's l1: 6.87348
[16000]	valid_0's l1: 6.85922
[18000]	valid_0's l1: 6.83366
[20000]	valid_0's l1: 6.81142
[22000]	valid_0's l1: 6.79577
[24000]	valid_0's l1: 6.78451
[26000]	valid_0's l1: 6.7754
[28000]	valid_0's l1: 6.76649
[30000]	valid_0's l1: 6.75491
[32000]	valid_0's l1: 6.74775
[34000]	valid_0's l1: 6.7389
[36000]	valid_0's l1: 6.72855
[38000]	valid_0's l1: 6.72033
[40000]	valid_0's l1: 6.7081
[42000]	valid_0's l1: 6.70213
[44000]	valid_0's l1: 6.68349
[46000]	valid_0's l1: 6.67481
[48000]	valid_0's l1: 6.65
[50000]	valid_0's l1: 6.64377
[52000]	valid_0's l1: 6.64025
[54000]	valid_0's l1: 6.63066
[56000]	valid_0's l1: 6.62688
[58000]	valid_0's l1: 6.62418
[60000]	valid_0's l1: 6.62197
[62000]	valid_0's l1: 6.61868
[64000]	valid_0's 

In [5]:
from xgboost import XGBRegressor

X = train[feature_cols]
y = train[TARGET]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_xgb = XGBRegressor(
    n_estimators          = 100000,
    learning_rate         = 0.05,
    max_depth             = 8,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.4,
    reg_lambda            = 1.0,
    objective             = 'reg:absoluteerror',
    tree_method           = 'hist',
    device                = 'cuda',
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 300,  # 여기로 이동
)

model_xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=5000,
)

best_iter = model_xgb.best_iteration
print(f'Best iteration: {best_iter}')
print(f'Validation MAE: {np.mean(np.abs(y_val - model_xgb.predict(X_val))):.4f}')

[0]	validation_0-mae:14.22634
[5000]	validation_0-mae:7.45297
[5828]	validation_0-mae:7.44577
Best iteration: 5528


c:\Users\user\anaconda3\envs\a\lib\site-packages\xgboost\core.py:751: UserWarning: [14:48:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Validation MAE: 7.4456


In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

xgb_params = {
    'n_estimators'         : 5528,
    'learning_rate'        : 0.05,
    'max_depth'            : 8,
    'subsample'            : 0.8,
    'colsample_bytree'     : 0.8,
    'reg_alpha'            : 0.4,
    'reg_lambda'           : 1.0,
    'objective'            : 'reg:absoluteerror',
    'tree_method'          : 'hist',
    'device'               : 'cuda',
    'random_state'         : 42,
    'n_jobs'               : -1,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'── Fold {fold+1} ──')
    X_tr,  y_tr  = X.iloc[tr_idx],  y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = XGBRegressor(**xgb_params)
    model.fit(X_tr, y_tr, verbose=False)

    joblib.dump(model, f'C:/Users/user/dacon/Smart_Logistics/models/xgb_fold{fold+1}.pkl')

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(test[feature_cols]) / 5

    fold_mae = np.mean(np.abs(y_val - model.predict(X_val)))
    print(f'Fold {fold+1} MAE: {fold_mae:.4f}')

oof_mae = np.mean(np.abs(y - oof_preds))
print(f'\nOOF MAE: {oof_mae:.4f}')

── Fold 1 ──
Fold 1 MAE: 7.3839
── Fold 2 ──
Fold 2 MAE: 7.4618
── Fold 3 ──
Fold 3 MAE: 7.3181
── Fold 4 ──
Fold 4 MAE: 7.5670
── Fold 5 ──
Fold 5 MAE: 7.4612

OOF MAE: 7.4384


In [7]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': test_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v12_xgb_kfold.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   15.303447
1  TEST_000001                   14.876473
2  TEST_000002                   18.193250
3  TEST_000003                   17.753496
4  TEST_000004                   16.659978
